## Carregamento dos dados

Este notebook carrega os dados em `data/ds003626/derivatives`, redimensiona-os para o formato (classes, trials, channels, time points) e persiste os dados formatados como arrays numpy em `data/processed`.

Os dados de cada participante estão separados da seguinte forma:
- **X-EEG**: Sinais EEG [trials × channels × time points]
- **X-EXG**: Sinais EOG e EMG [trials x channels x time points]
- **Y**: Rótulos especificando um evento [trials × 4]
  - Coluna 0: Timestamp
  - Coluna 1: Classe (0=up, 1=down, 2=right, 3=left)
  - Coluna 2: Condição (0=pronounced, 1=inner speech, 2=visualized)
  - Coluna 3: Sessão

In [ ]:
# Imports
import sys
import mne
import numpy as np
import pandas as pd
from pathlib import Path

# Add the Inner Speech Dataset repository to the system path
sys.path.append(str(Path.cwd().parent / "src" / "Python_Processing"))

from lib.data_extractions import load_events
from lib.utils import sub_name

# List with every subject (S1-S10)
N_S = list(range(1, 11))

# sub-09 ses-01 and sub-10 ses-03 are truncated in the OpenNeuro datasets
EXCLUDE = {9, 10}
SUBJECTS = [i for i in N_S if i not in EXCLUDE]

# Datasets downloaded in notebook 01-download
root_dir = Path.cwd().parent / "data" / "ds003626"

# Directory for the np arrays generated in this notebook
processed_dir = Path.cwd().parent / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

### Salvando dados em arrays numpy

In [ ]:
# Loads all data from a single subject
def load_subject(root_dir: Path, n_s: int, datatype="eeg"):
    num_s = sub_name(n_s)
    x, y = [], []

    for n_b in (1, 2, 3):
        file_name = f"{root_dir}/derivatives/{num_s}/ses-0{n_b}/{num_s}_ses-0{n_b}_{datatype}-epo.fif"

        try:
            ep = mne.read_epochs(file_name, verbose="WARNING")
            data = ep.get_data()
        except Exception as e:
            print(f"Skipping {num_s} ses-0{n_b}: {type(e).__name__}")
            continue

        ev = pd.DataFrame(load_events(root_dir, n_s, n_b),
                          columns=["timestamp", "direction", "condition", "session"])
        
        ev["subject"] = n_s

        x.append(data)
        y.append(ev)

        ch_names, times = ep.ch_names, ep.times
        
        print(f"S_{n_s} ses-0{n_b}: {len(data)} trials")

    X = np.concatenate(x)
    Y = pd.concat(y, ignore_index=True)

    return X, Y, ch_names, times

In [ ]:
# Loads each subject's data and saves it as np arrays

for s in SUBJECTS:
    X_eeg, Y_eeg, ch_names_eeg, times_eeg = load_subject(root_dir, s, 'eeg')
    X_exg, Y_exg, ch_names_exg, times_exg = load_subject(root_dir, s, 'exg')

    # Both Y_egg and Y_exg are the same, we can persist either one
    Y = Y_eeg[["timestamp", "direction", "condition", "session"]].to_numpy()

    print(f"S_{s}: EEG = {X_eeg.shape}, EXG = {X_exg.shape}, Y = {Y.shape}\n")

    np.save(processed_dir / f"S{s}_eeg", X_eeg)
    np.save(processed_dir / f"S{s}_exg", X_exg)
    np.save(processed_dir / f"S{s}_y", Y)
    np.save(processed_dir / f"S{s}_eeg_channels", np.asarray(ch_names_eeg))
    np.save(processed_dir / f"S{s}_exg_channels", np.asarray(ch_names_exg))
    np.save(processed_dir / f"S{s}_times", np.asarray(times_eeg))

Os arquivos salvos são utilizados para construir um único array numpy, separando as classes e trials em duas dimensões diferentes. Serão considerados os dados sob a condição 1 - Inner Speech para todas as classes disponíveis.

Também serão salvos duas estruturas auxiliares:
- subj: vetor para a identificação de um participante. Formato (classes, trials);
- ses: vetor para a identificação de uma sessão. Formato (classes, trials).

In [ ]:
# Saves every subject data in a single file with the format (classes, trials, channels, timepoints)

CLASSES = {0: "up", 1: "down", 2: "right", 3: "left"}
COND = 1 # 0 = pronounced, 1 = inner, 2 = visualized

# One array per class
class_eeg = {c: [] for c in CLASSES}
class_exg = {c: [] for c in CLASSES}
class_sbj = {c: [] for c in CLASSES}
class_ses = {c: [] for c in CLASSES}

for s in SUBJECTS:
    X_eeg = np.load(processed_dir / f"S{s}_eeg.npy")
    X_exg = np.load(processed_dir / f"S{s}_exg.npy")
    Y = np.load(processed_dir / f"S{s}_y.npy")

    for c in CLASSES:
        # Find the correct indices that match the current class
        idx = np.where((Y[:, 1] == c) & (Y[:, 2] == COND))[0]

        # Append data from the current class to the appropriate class arrays
        class_eeg[c].append(np.asarray(X_eeg[idx], dtype=np.float32))
        class_exg[c].append(np.asarray(X_exg[idx], dtype=np.float32))
        class_sbj[c].append(np.full(len(idx), s))
        class_ses[c].append(Y[idx, 3].astype(int))

# Concatenate all classes arrays into one, merging on trials
class_eeg = {c: np.concatenate(v) for c, v in class_eeg.items()}
class_exg = {c: np.concatenate(v) for c, v in class_exg.items()}
class_sbj = {c: np.concatenate(v) for c, v in class_sbj.items()}
class_ses = {c: np.concatenate(v) for c, v in class_ses.items()}

counts = {CLASSES[c]: len(v) for c, v in class_eeg.items()}
print(f"Counts for each class: {counts}")

# Stack on classes to create the new class dimension (from [trials x channels x time points] to [classes x trials x channels x time points])
X_eeg = np.stack([class_eeg[c] for c in CLASSES])
X_exg = np.stack([class_exg[c] for c in CLASSES])
subj = np.stack([class_sbj[c] for c in CLASSES])
ses = np.stack([class_ses[c] for c in CLASSES])

np.save(processed_dir / "X_eeg.npy", X_eeg)
np.save(processed_dir / "X_exg.npy", X_exg)
np.save(processed_dir / "subj.npy", subj)
np.save(processed_dir / "ses.npy", ses)

print(f"Saved EEG in format {X_eeg.shape} and EXG in format {X_exg.shape}")